<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-06-rag-systems/lesson-6.9-multimodal-rag/notebooks/exercises-6.9.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 6.9 — Multimodal RAG
Netsetos GenAI Course v2.0 | Module 6

Image/table/audio/video RAG, ColPali, Document AI, multimodal embeddings, production patterns


In [ ]:
# pip install byaldi unstructured openai llama-index faster-whisper -q
print('Ready for Multimodal RAG')


## Ex 1: Image-as-Text Strategy


In [ ]:
import base64
# from openai import OpenAI
# client = OpenAI()

def describe_image(image_path):
    with open(image_path, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    # response = client.chat.completions.create(
    #     model='gpt-4o',
    #     messages=[{'role':'user','content':[
    #         {'type':'text','text':'Describe this image in detail for RAG indexing.'},
    #         {'type':'image_url','image_url':{'url':f'data:image/jpeg;base64,{b64}'}}
    #     ]}]
    # )
    # return response.choices[0].message.content
    return 'Chart showing Q3 revenue of $4.2M, up 15% from Q2'

print('Image description:', describe_image('chart.png'))
print('\nThis text description gets embedded alongside document text')
print('Cost: GPT-4o low-detail = 85 tokens/image ($0.0004)')


## Ex 2: ColPali/ColQwen with Byaldi


In [ ]:
# from byaldi import RAGMultiModalModel
# RAG = RAGMultiModalModel.from_pretrained('vidore/colqwen2-v1.0')
# RAG.index(input_path='docs/', index_name='my_index')
# results = RAG.search('What is the revenue breakdown?', k=5)
# for r in results:
#     print(f'Doc {r.doc_id}, Page {r.page_num}, Score: {r.score:.3f}')

print('ColPali/ColQwen pipeline:')
print('  1. Page screenshot → SigLIP vision encoder')
print('  2. 1,024 patch embeddings (128-dim each)')
print('  3. Query → text encoder → token embeddings')
print('  4. MaxSim: for each query token, find best patch → sum')
print()
print('Models:')
for m, s in [('ColPali','81.3 nDCG@5 ViDoRe v1'),('ColQwen2','86.6'),
             ('ColQwen2.5','0.597 ViDoRe v2'),('ColSmol-256M','80.1')]:
    print(f'  {m:20s}: {s}')


## Ex 3: Unstructured.io Document Processing


In [ ]:
# from unstructured.partition.pdf import partition_pdf
# elements = partition_pdf(
#     filename='report.pdf',
#     strategy='hi_res',
#     infer_table_structure=True,
#     extract_images_in_pdf=True,
# )
# for el in elements:
#     print(f'{el.category}: {str(el)[:100]}')

print('Unstructured strategies:')
for s, desc in [('Fast','PDFMiner text-only'),
    ('Hi_Res','YOLOX layout + OCR + PDFMiner merged'),
    ('VLM','Pages sent to GPT-4o/Claude'),
    ('Auto','Routes each page optimally')]:
    print(f'  {s:8s}: {desc}')
print('\nLayout elements: Title, NarrativeText, Table, Image, Caption, Formula')


## Ex 4: Multimodal Embeddings — Nomic Embed Vision


In [ ]:
# from nomic import embed
# text_embeddings = embed.text(['revenue chart', 'Q3 results'])
# image_embeddings = embed.image(['chart.png', 'diagram.png'])
# # Both live in SAME vector space — cross-modal search works!

print('Multimodal embedding models:')
for m, feat in [
    ('SigLIP 2','Dynamic resolution, 86M-1B params, Google'),
    ('Nomic Embed Vision','Backward-compatible, 92M, Matryoshka 64-768'),
    ('BGE-VL','Universal cross-modal, MIT, +8.1% CIRCO'),
    ('CLIP/OpenCLIP','Original joint space, strong baseline'),
]: print(f'  {m:20s}: {feat}')
print('\n3 retrieval strategies:')
print('  1. Separate indices per modality → fuse via RRF')
print('  2. Unified index (Nomic/BGE-VL) → simpler')
print('  3. Text grounding (convert all → text) → most mature')


## Ex 5: Audio RAG with Whisper


In [ ]:
# from faster_whisper import WhisperModel
# model = WhisperModel('large-v3-turbo', device='cuda')
# segments, info = model.transcribe('lecture.mp3', word_timestamps=True)
# chunks = []
# for seg in segments:
#     chunks.append({
#         'text': seg.text, 'start': seg.start, 'end': seg.end
#     })

print('Audio RAG pipeline:')
print('  1. Whisper large-v3-turbo (8× faster, 216× real-time)')
print('  2. Word-level timestamps')
print('  3. Chunk by sentence/topic boundaries')
print('  4. Embed text + store timestamp metadata')
print('  5. Query → retrieve chunk → return text + audio position')
print()
print('CLAP: audio equivalent of CLIP')
print('  Text→audio search: "find glass breaking" → exact clip')


## Ex 6: Video RAG — 3-Stream Pipeline


In [ ]:
print('Video RAG 3-stream pipeline:')
print()
print('Stream 1: ASR (biggest improvement)')
print('  Whisper → word-level timestamps → text chunks')
print()
print('Stream 2: Key frames')
print('  Extract every 5-10s or shot-based')
print('  VLM description: Gemini 2.0 Flash ($0.075/MTok)')
print('  33× cheaper than GPT-4o for bulk processing')
print()
print('Stream 3: On-screen OCR')
print('  Slides, text overlays, captions')
print()
print('Alignment: all 3 streams indexed by timestamp')
print('Query → retrieve text chunk → return video position + context')
print()
print('NeurIPS 2025: surpasses Gemini-1.5-Pro with ~1.9K aux tokens')


## Ex 7: Production Cost Optimization


In [ ]:
print('Cost comparison per page/image:')
for method, cost in [
    ('OCR + text embedding', '$0.001-0.01'),
    ('VLM summarization (GPT-4o)', '$0.01-0.05'),
    ('VLM summarization (Gemini Flash)', '$0.003-0.01'),
    ('ColPali embedding (GPU)', 'GPU compute, no API'),
]: print(f'  {method:35s}: {cost}')
print()
print('Optimization strategies:')
print('  1. Gemini 2.0 Flash for bulk VLM (33× cheaper than GPT-4o)')
print('  2. Content-hash cache in Redis (60-80% hit rates)')
print('  3. Pre-process VLM at ingestion, NOT query time')
print('  4. GPT-4o low-detail: 85 tokens/image (flat)')
print('  5. Unstructured Auto: cheapest adequate strategy per page')
print('  6. Light-ColPali: 98.2% quality at 11.8% storage')


## Ex 8: India Document OCR


In [ ]:
import re

def validate_aadhaar(text):
    pattern = r'\b\d{4}\s?\d{4}\s?\d{4}\b'
    matches = re.findall(pattern, text)
    return matches

def validate_pan(text):
    pattern = r'\b[A-Z]{5}[0-9]{4}[A-Z]\b'
    matches = re.findall(pattern, text)
    return matches

test = 'Name: राहुल कुमार, Aadhaar: 1234 5678 9012, PAN: ABCDE1234F'
print('Aadhaar:', validate_aadhaar(test))
print('PAN:', validate_pan(test))
print()
print('Devanagari OCR comparison:')
for tool, cer in [('Google Cloud Vision','14.6% CER (handwritten)'),
    ('EasyOCR','46.8%'),('Tesseract','85.5% (unusable)')]:
    print(f'  {tool:25s}: {cer}')
print()
print('DigiLocker: 250M+ users, structured XML, skip OCR entirely')


---
## Recap
Multimodal RAG handles images, tables, audio, and video alongside text. Three image strategies: image-as-text (simplest), multimodal embeddings (CLIP/Nomic/BGE-VL), direct vision (highest quality). ColPali/ColQwen (ICLR 2025) eliminates OCR→chunk→embed with vision-first retrieval — page screenshots → 1,024 patch embeddings → MaxSim scoring. ColQwen2.5: 0.597 nDCG@5. Document AI: Unstructured (production ETL), DocTR (PyTorch OCR), LlamaParse v2 (130+ formats). Audio RAG: Whisper large-v3-turbo (8× faster) + timestamps. Video RAG: 3-stream (ASR + frames + OCR) aligned by timestamp. Production: Gemini Flash for bulk VLM (33× cheaper), content-hash caching, pre-process at ingestion. India: Google Vision for Devanagari (14.6% vs Tesseract 85.5%), DigiLocker API shortcut, Bhashini Mozhi dataset, Qwen3-Embedding-8B for multilingual.
